# Blocking Jacobi with CUDA-aware MPI

This notebook explains the example in [src/04-jacobi-blocking.cu](../src/04-jacobi-blocking.cu) and connects it to the chapter in [docs/source/04-jacobi.rst](../docs/source/04-jacobi.rst).

## The problem we are solving

We want to solve a 2-D steady-state heat equation over a large grid. The update rule is:

$$
T_{i,j}^{new} = \frac{1}{4}\left(T_{i-1,j} + T_{i+1,j} + T_{i,j-1} + T_{i,j+1}\right)
$$

This is the five-point Jacobi stencil. The challenge is that the global grid is too large for one GPU or one rank, so we split it across MPI ranks.

Each rank owns a slice of the domain, but to update the cells near its boundary it must know the neighboring values from the adjacent rank. We therefore keep ghost rows at the top and bottom of the local array and exchange them with the neighboring ranks every iteration.

The goal is to:
- keep the work on the GPU
- avoid host staging
- keep communication between ranks correct and synchronized
- update the full grid repeatedly until a fixed number of Jacobi iterations is reached

## Why ghost rows are needed

A local rank does not own the neighboring rank's cells, but the stencil needs them at the boundary. The solution is a halo exchange:

- each rank sends its outermost real row to its neighbor
- each rank receives the neighbor's outermost row into a ghost row

That way the local stencil can read values as if a larger local domain existed, even though the data is physically split across ranks.

![Jacobi row decomposition](../docs/source/images/05-jacobi-row-decomposition.png)

This figure is the same diagram used in the documentation chapter and shows the owned rows, ghost rows, and the five-point stencil.

## 1. High-level structure of the code

The program does the following in order:

1. Initialize MPI and choose a GPU for each rank.
2. Split the global domain across ranks.
3. Allocate two device arrays, `u` and `v`.
4. Initialize the boundary conditions.
5. Repeat for `iters` iterations:
   - exchange halo rows via MPI
   - run the Jacobi update kernel on the GPU
   - swap old/new arrays
6. Measure time and report a checksum.

This is the basic pattern of a CUDA-aware distributed stencil solver.

## 2. MPI setup and device selection

The code starts with MPI setup:

```cpp
MPI_Init(&argc, &argv);
int rank, size;
MPI_Comm_rank(MPI_COMM_WORLD, &rank);
MPI_Comm_size(MPI_COMM_WORLD, &size);
```

Then it calls:

```cpp
select_device(MPI_COMM_WORLD);
```

This helper picks a GPU based on the rank's node-local position instead of the global MPI rank. That is important when multiple ranks share the same node. It uses a shared communicator and chooses:

```cpp
device = local_rank % device_count;
```

This ensures each rank on the same node gets a different GPU when possible.

## 3. Domain decomposition and chunk sizes

The user supplies arguments like:

```cpp
int nx = argc > 1 ? atoi(argv[1]) : 4096;
int ny = argc > 2 ? atoi(argv[2]) : 4096;
int iters = argc > 3 ? atoi(argv[3]) : 200;
```

Then it checks that the vertical dimension is divisible by the number of ranks:

```cpp
if (ny % size || nx < 3 || ny / size < 3) {
```

Each rank gets a local chunk of height:

```cpp
int rows = ny / size;
int first = rank * rows;
```

So the global index of the local row is mapped using:

```cpp
int gy = first + y - 1;
```

This allows the boundary condition code to correctly identify whether a point is on the global domain edge.

The neighbor ranks are:

```cpp
int up = rank ? rank - 1 : MPI_PROC_NULL;
int down = rank < size - 1 ? rank + 1 : MPI_PROC_NULL;
```

The top and bottom ranks are connected through `up` and `down`, and the global edges use `MPI_PROC_NULL` so no send/receive occurs there.

## 4. Buffer layout: real rows plus ghost rows

The local array size is:

```cpp
size_t bytes = (size_t)(rows + 2) * nx * sizeof(double);
```

The extra two rows are the ghost rows:

- row 0: top ghost row
- rows + 1: bottom ghost row

The real rows are stored in between. This is why the code uses the local array as:

```cpp
u + nx
u + rows * nx
u + (rows + 1) * nx
```

Those are pointer offsets into the same device array, not separate arrays.

The buffer holds one local domain slice, not just the owned cells.

In [ ]:
# A quick conceptual view of the local memory layout
rows = 5
nx = 6

def print_layout(rows, nx):
    # Top ghost, owned rows, bottom ghost
    layout = ["top ghost"] + [f"owned {i}" for i in range(rows)] + ["bottom ghost"]
    print(f"rows: {rows}, nx: {nx}")
    print(layout)

print_layout(rows, nx)

## 5. Initialization kernel

The `init` kernel sets the boundary values:

```cpp
__global__ void init(double *u, int rows, int nx, int ny, int first)
{
  int x = blockIdx.x * blockDim.x + threadIdx.x,
      y = blockIdx.y * blockDim.y + threadIdx.y;
  if (x < nx && y < rows + 2) {
    int gy = first + y - 1;
    u[y * nx + x] =
        (x == 0 || x == nx - 1 || gy == 0 || gy == ny - 1) ? 1. : 0.;
  }
}
```

This does two things:

- marks the global outer boundaries as `1.0`
- marks all interior points as `0.0`

The condition uses `gy = first + y - 1` to convert the local row index back to the global row number. That is necessary so a rank on the global boundary can detect whether it touches the outside of the whole problem.

Both arrays are initialized this way:

```cpp
init<<<g, b>>>(u, rows, nx, ny, first);
init<<<g, b>>>(v, rows, nx, ny, first);
```

## 6. CUDA launch dimensions

The code creates:

```cpp
dim3 b(32, 8);
dim3 g((nx + 31) / 32, (rows + 9) / 8);
```

This means:

- `b` is the block size: `32 x 8` threads
- `g` is the grid size: enough blocks to cover the whole local domain

Each block has `32 * 8 = 256` threads. The grid dimensions are rounded up to ensure every element is covered, including the ghost rows.

This is a classic CUDA pattern for a 2D array: tile the grid and map each thread to one array element.

## 7. Jacobi update kernel

The stencil update is implemented in:

```cpp
__global__ void step(const double *u, double *v, int nx, int begin, int end)
{
  int x = blockIdx.x * blockDim.x + threadIdx.x + 1,
      y = blockIdx.y * blockDim.y + threadIdx.y + begin;
  if (x < nx - 1 && y <= end) {
    v[y * nx + x] = .25 * (u[y * nx + x - 1] + u[y * nx + x + 1] +
                           u[(y - 1) * nx + x] + u[(y + 1) * nx + x]);
  }
}
```

The update is:

$$
V_{i,j} = \frac{1}{4}(U_{i-1,j} + U_{i+1,j} + U_{i,j-1} + U_{i,j+1})
$$

This uses the four neighbors in the north, south, east, and west directions.

The code intentionally skips the left boundary by starting `x` at `1`, and also skips the top/bottom ghost rows by using `begin` and `end`.

## 8. MPI halo exchange

The core communication is here:

```cpp
MPI_CHECK(MPI_Sendrecv(
    u + nx,                               /* first real row to send upward */
    nx,                                   /* one full row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    up,                                   /* rank above */
    10,                                   /* tag for upward exchange */
    u + (rows + 1) * nx,                  /* bottom ghost row to receive from below */
    nx,                                   /* one full row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    down,                                 /* rank below */
    10,                                   /* matching tag for the below neighbor */
    MPI_COMM_WORLD,                       /* all ranks in the solver */
    MPI_STATUS_IGNORE                     /* ignore the MPI status */
));
```

This says:

- send the first real row upward to the rank above
- receive the below neighbor's first row into the bottom ghost row

The second call does the opposite direction:

```cpp
MPI_CHECK(MPI_Sendrecv(
    u + rows * nx,                        /* last real row to send downward */
    nx,                                   /* one full row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    down,                                 /* rank below */
    11,                                   /* tag for downward exchange */
    u,                                    /* top ghost row to receive from above */
    nx,                                   /* one full row of nx doubles */
    MPI_DOUBLE,                           /* data type */
    up,                                   /* rank above */
    11,                                   /* matching tag for the above neighbor */
    MPI_COMM_WORLD,                       /* all ranks in the solver */
    MPI_STATUS_IGNORE                     /* ignore the MPI status */
));
```

This updates the top and bottom ghost rows in a blocking, synchronous way.

This is the "blocking" version because the rank must complete the communication before proceeding to compute the next iteration.

## 9. Why the code swaps `u` and `v`

The program alternates between two arrays:

- `u`: current values
- `v`: next values

This is the standard Jacobi pattern. Each iteration reads from `u` and writes to `v`, then swaps them:

```cpp
step<<<work, b>>>(u, v, nx, begin, end);
CUDA_CHECK(cudaDeviceSynchronize());
std::swap(u, v);
```

This avoids overwriting the old values before all neighboring reads are complete.

## 10. Timing and reporting

After the loop finishes, the code measures the per-rank runtime:

```cpp
double local = MPI_Wtime() - t, elapsed;
MPI_Reduce(&local, &elapsed, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);
```

The `MPI_MAX` reduction gives the maximum rank time, which is the relevant wall-clock time for the parallel application.

Then it samples the center of the local domain:

```cpp
CUDA_CHECK(cudaMemcpy(&sample, u + (rows / 2) * nx + nx / 2, sizeof(double),
                      cudaMemcpyDeviceToHost));
```

and sums it across all ranks:

```cpp
MPI_Reduce(&sample, &checksum, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
```

This sample is a diagnostic check, not a proof of correctness. It is a lightweight way to see that the solver is progressing and to compare runs.

Finally, rank 0 prints the overall result:

```cpp
printf("blocking Jacobi: %dx%d, %d ranks, %d iterations, %.3f s, sample sum %.12e\n",
       nx, ny, size, iters, elapsed, checksum);
```

## 11. Summary of the algorithm

The code realizes this pattern:

1. split the global grid across MPI ranks
2. allocate a local array with ghost rows
3. initialize the boundary conditions
4. exchange ghost rows with neighbors
5. run the Jacobi update on the GPU
6. swap the arrays and repeat
7. report the performance and a small diagnostic checksum

The blocking behavior is simple and correct, but it means communication and computation are serialized: no rank can advance to the next update until the halo exchange finishes.

## 12. Image from the chapter

The documentation chapter also emphasizes the same decomposition and communication structure:

![Jacobi decomposition figure](../docs/source/images/05-jacobi-row-decomposition.png)

This notebook intentionally mirrors that figure and explains the code that implements the same concept in CUDA-aware MPI.